In [ ]:
### Config stuff

In [ ]:
import random
import pyspark
from pyspark.sql import SparkSession, functions
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

In [ ]:
spark = cc.startLocalCluster("dimLock")
spark.getActiveSession()


In [ ]:
cc.set_connectionProfile("velodb")

df_locks = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "locks") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "lockid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100) \
    .load()

df_stations = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

In [ ]:
print(cc.create_jdbc())

In [ ]:
# Join locks + stations
df_lockdim = df_locks.alias("l") \
    .join(df_stations.alias("s"), col("l.stationid") == col("s.stationid"), "left") \
    .select(
        col("l.lockid").alias("lock_id"),
        col("l.stationid").alias("station_id"),
        col("l.stationlocknr").alias("station_nr"),
        col("s.street"),
        col("s.number"),
        col("s.zipcode"),
        col("s.district"),
        col("s.gpscoord").alias("gps_coord")
    )


In [ ]:
#A dapt this:
# Step 4: Inspect schema and preview
df_lockdim.printSchema()
df_lockdim.show()
df_lockdim.write.format("delta").mode("overwrite").saveAsTable("LockDim")





## Delete the spark session

In [ ]:
spark.stop()